# Datathon FIAP — Machine Learning 

Este notebook refaz a etapa de machine learning a partir dos 3 CSVs originais e segue uma lógica alinhada ao enunciado:

- usar o histórico para prever risco de defasagem;
- definir o alvo de forma coerente como aluno defasado em 2024;
- evitar vazamento de dados, usando apenas informações históricas como features;
- tratar explicitamente a ausência de IPP em 2022, conforme orientação do professor.



In [1]:

from pathlib import Path
import re
import json
import joblib
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score,
    fbeta_score, confusion_matrix, classification_report, precision_recall_curve
)
from sklearn.inspection import permutation_importance
from sklearn.neighbors import KNeighborsRegressor

warnings.filterwarnings("ignore")

BASE_DIR = Path(r"C:\Users\gfurt\ML_Projects\FIAP\Datathon\datathon-passos-magicos-fase5\data\processed")
FILE_2022 = BASE_DIR / "pede_2022_tratado.csv"
FILE_2023 = BASE_DIR / "pede_2023_tratado.csv"
FILE_2024 = BASE_DIR / "pede_2024_tratado.csv"

df22_raw = pd.read_csv(FILE_2022)
df23_raw = pd.read_csv(FILE_2023)
df24_raw = pd.read_csv(FILE_2024)

print("2022:", df22_raw.shape)
print("2023:", df23_raw.shape)
print("2024:", df24_raw.shape)


2022: (860, 43)
2023: (1014, 48)
2024: (1156, 50)


## 1. Padronização de colunas e funções auxiliares

In [2]:

def normalize_colname(col: str) -> str:
    col = re.sub(r"[^0-9A-Za-z_À-ÿ]+", "_", str(col)).strip("_")
    return col

def parse_br_number(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none"}:
        return np.nan
    s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except:
        return np.nan

def prep_2022(df):
    out = df.copy()
    out = out.rename(columns={
        "Ano nasc": "Ano_nasc",
        "Idade 22": "Idade",
        "INDE 22": "INDE",
        "Pedra 22": "Pedra",
        "Matem": "Mat",
        "Portug": "Por",
        "Inglês": "Ing",
        "Fase ideal": "Fase_Ideal",
        "Defas": "Defasagem",
    })
    out.columns = [normalize_colname(c) for c in out.columns]
    out["IPP"] = np.nan  # criado explicitamente por orientação do professor
    return out

def prep_2023(df):
    out = df.copy()
    out = out.rename(columns={
        "Nome Anonimizado": "Nome",
        "Data de Nasc": "Data_Nasc",
        "INDE 2023": "INDE",
        "Pedra 2023": "Pedra",
        "Fase Ideal": "Fase_Ideal",
    })
    out.columns = [normalize_colname(c) for c in out.columns]
    return out

def prep_2024(df):
    out = df.copy()
    out = out.rename(columns={
        "Nome Anonimizado": "Nome",
        "Data de Nasc": "Data_Nasc",
        "INDE 2024": "INDE",
        "Pedra 2024": "Pedra",
        "Fase Ideal": "Fase_Ideal",
    })
    out.columns = [normalize_colname(c) for c in out.columns]
    return out

df22 = prep_2022(df22_raw)
df23 = prep_2023(df23_raw)
df24 = prep_2024(df24_raw)

print("Colunas 2022:", len(df22.columns))
print("Colunas 2023:", len(df23.columns))
print("Colunas 2024:", len(df24.columns))


Colunas 2022: 43
Colunas 2023: 48
Colunas 2024: 50


## 2. Conversão de variáveis numéricas

Os arquivos usam vários números no padrão brasileiro, como 7,278 ou 5,000.  
Aqui transformamos essas colunas em valores numéricos.


In [3]:

numeric_candidates = [
    "Ano_nasc","Idade","Ano_ingresso","Fase","INDE","Cg","Cf","Ct","N_Av",
    "IAA","IEG","IPS","IPP","IDA","Mat","Por","Ing","IPV","IAN","Defasagem"
]

for df in [df22, df23, df24]:
    for col in numeric_candidates:
        if col in df.columns:
            df[col] = df[col].map(parse_br_number)

print(df22[numeric_candidates[:8]].head(2))


   Ano_nasc  Idade  Ano_ingresso  Fase   INDE     Cg    Cf    Ct
0    2003.0   19.0        2016.0   7.0  5.783  753.0  18.0  10.0
1    2005.0   17.0        2017.0   7.0  7.055  469.0   8.0   3.0


## 3. Reconstrução explícita de IPP_2022 com KNN

Como IPP não existe em 2022, não dá para usar KNNImputer puro diretamente, porque a coluna inteira está vazia nesse ano.  
Então foi usada uma abordagem KNN supervisionada:

1. juntamos 2023 e 2024, onde IPP existe;
2. treinamos um KNeighborsRegressor para aprender o padrão do IPP;
3. aplicamos esse modelo nas linhas de 2022 para estimar IPP_2022.

Na prática, isso implementa a ideia sugerida pelo professor de preencher o campo com base nas abas de 2023 e 2024.


In [4]:

impute_features = [c for c in numeric_candidates if c in df23.columns and c in df24.columns and c not in {"IPP"}]

imp_train = pd.concat(
    [df23[impute_features + ["IPP"]], df24[impute_features + ["IPP"]]],
    ignore_index=True
)

usable_impute_features = [c for c in impute_features if not imp_train[c].isna().all()]
median_map = imp_train[usable_impute_features].median()

imp_train2 = imp_train[usable_impute_features + ["IPP"]].copy()
imp_train2[usable_impute_features] = imp_train2[usable_impute_features].fillna(median_map)
imp_train2 = imp_train2[imp_train2["IPP"].notna()].copy()

knn_reg_ipp = KNeighborsRegressor(n_neighbors=15, weights="distance")
knn_reg_ipp.fit(imp_train2[usable_impute_features], imp_train2["IPP"])

X22_imp = df22[usable_impute_features].copy().fillna(median_map)
df22["IPP"] = knn_reg_ipp.predict(X22_imp)

print("IPP_2022 imputado com sucesso.")
print(df22["IPP"].describe())


IPP_2022 imputado com sucesso.
count    860.000000
mean       7.445358
std        0.470121
min        5.975566
25%        7.138006
50%        7.412422
75%        7.775342
max        8.551542
Name: IPP, dtype: float64


## 4. Construção da base longitudinal (wide)

Para machine learning supervisionado, cada aluno precisa ficar em uma linha.  
Então juntamos os anos por RA e deixamos as variáveis com sufixo do ano:

- IEG_2022, IEG_2023, IEG_2024
- IPP_2022, IPP_2023, IPP_2024
- etc.


In [5]:

ra22 = set(df22["RA"])
ra23 = set(df23["RA"])
ra24 = set(df24["RA"])

print("Alunos em 2022:", len(ra22))
print("Alunos em 2023:", len(ra23))
print("Alunos em 2024:", len(ra24))
print("Interseção 2023 e 2024:", len(ra23 & ra24))
print("Interseção 2022, 2023 e 2024:", len(ra22 & ra23 & ra24))

def suffix_except_ra(df, year):
    return df.rename(columns={c: (c if c == "RA" else f"{c}_{year}") for c in df.columns})

df22_w = suffix_except_ra(df22, 2022)
df23_w = suffix_except_ra(df23, 2023)
df24_w = suffix_except_ra(df24, 2024)

# Base principal: alunos com histórico em 2023 e alvo observado em 2024.
df_model = (
    df23_w
    .merge(df24_w, on="RA", how="inner")
    .merge(df22_w, on="RA", how="left")
)

print("Base wide final:", df_model.shape)
display(df_model.head(2))


Alunos em 2022: 860
Alunos em 2023: 1014
Alunos em 2024: 1156
Interseção 2023 e 2024: 765
Interseção 2022, 2023 e 2024: 468
Base wide final: (765, 139)


,RA,Fase_2023,INDE_2023,Pedra_2023,Turma_2023,Nome_2023,Data_Nasc_2023,Idade_2023,Gênero_2023,Ano_ingresso_2023,...,Indicado_2022,Atingiu_PV_2022,IPV_2022,IAN_2022,Fase_Ideal_2022,Defasagem_2022,Destaque_IEG_2022,Destaque_IDA_2022,Destaque_IPV_2022,IPP_2022
0,RA-861,NaN,9.31095,Topázio,ALFA A - G0/G1,Aluno-861,6/17/2015,8.0,Feminino,2023.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RA-862,NaN,8.22120,Topázio,ALFA A - G0/G1,Aluno-862,5/31/2014,9.0,Masculino,2023.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Definição do target

Pelo enunciado, o objetivo é prever risco de defasagem.

Aqui a decisão foi:

- target = 1 se Defasagem_2024 < 0
- target = 0 caso contrário

Essa definição é coerente com a lógica de que valores negativos em Defasagem indicam aluno atrasado/defasado.


In [6]:

df_model["target_risco_defasagem_2024"] = (df_model["Defasagem_2024"] < 0).astype(int)

target_dist = (
    df_model["target_risco_defasagem_2024"]
    .value_counts(normalize=True)
    .rename_axis("classe")
    .reset_index(name="proporcao")
)

display(target_dist)


,classe,proporcao
0,0,0.597386
1,1,0.402614


## 6. Seleção de features e prevenção de leakage

Para prever o risco em 2024, não usamos variáveis de 2024 como entrada.  
Assim, as features vêm apenas do histórico de 2022 e 2023.

Também removemos colunas de identificação textual que não agregam à modelagem.


In [7]:

drop_cols = [
    "RA",
    "Nome_2022", "Nome_2023", "Nome_2024",
]

feature_cols = [
    c for c in df_model.columns
    if c not in drop_cols
    and c != "target_risco_defasagem_2024"
    and not c.endswith("_2024")
]

X = df_model[feature_cols].copy()
y = df_model["target_risco_defasagem_2024"].copy()

all_missing_cols = [c for c in X.columns if X[c].isna().all()]
X = X.drop(columns=all_missing_cols)

numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print("Total de features:", X.shape[1])
print("Numéricas:", len(numeric_cols))
print("Categóricas:", len(categorical_cols))


Total de features: 70
Numéricas: 35
Categóricas: 35


## 7. Divisão em treino, validação e teste

Aqui usamos:

- 70% treino
- 15% validação
- 15% teste

A escolha do melhor modelo e do melhor threshold é feita na validação, e o teste fica reservado para avaliação final.


In [8]:

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Treino:", X_train.shape, y_train.mean())
print("Validação:", X_valid.shape, y_valid.mean())
print("Teste:", X_test.shape, y_test.mean())


Treino: (535, 70) 0.40186915887850466
Validação: (115, 70) 0.40869565217391307
Teste: (115, 70) 0.4


## 8. Pipeline de pré-processamento e candidatos

Comparação entre:

- Logistic Regression
- Random Forest

Critério principal de escolha: PR-AUC na validação, porque o foco do problema é detectar o risco com boa recuperação dos casos positivos.


In [9]:

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
    ]
)

candidate_models = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
    ),
}

validation_results = []
fitted_models = {}

for model_name, estimator in candidate_models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", estimator),
    ])
    pipe.fit(X_train, y_train)
    valid_proba = pipe.predict_proba(X_valid)[:, 1]

    validation_results.append({
        "model": model_name,
        "roc_auc_valid": roc_auc_score(y_valid, valid_proba),
        "pr_auc_valid": average_precision_score(y_valid, valid_proba),
    })
    fitted_models[model_name] = pipe

validation_results_df = pd.DataFrame(validation_results).sort_values(
    by="pr_auc_valid", ascending=False
).reset_index(drop=True)

validation_results_df


,model,roc_auc_valid,pr_auc_valid
0,RandomForest,0.940864,0.92809
1,LogisticRegression,0.904255,0.85557


## 9. Escolha do melhor modelo e do threshold ótimo

Depois de escolher o melhor modelo na validação, ajustamos o threshold usando F2-score na validação.

O F2-score dá mais peso ao recall, o que faz sentido quando a prioridade é não deixar alunos em risco passarem despercebidos.


In [10]:

best_model_name = validation_results_df.loc[0, "model"]
best_pipe = fitted_models[best_model_name]

valid_proba = best_pipe.predict_proba(X_valid)[:, 1]
precision_vals, recall_vals, thresholds = precision_recall_curve(y_valid, valid_proba)

f2_scores = (5 * precision_vals[:-1] * recall_vals[:-1]) / (
    4 * precision_vals[:-1] + recall_vals[:-1] + 1e-12
)

best_threshold = float(thresholds[np.nanargmax(f2_scores)])

print("Melhor modelo:", best_model_name)
print("Threshold ótimo (validação):", round(best_threshold, 4))
print("Melhor F2 na validação:", round(float(np.nanmax(f2_scores)), 4))


Melhor modelo: RandomForest
Threshold ótimo (validação): 0.3562
Melhor F2 na validação: 0.8915


## 10. Avaliação final no conjunto de teste

In [11]:

test_proba = best_pipe.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

metrics_test = {
    "model": best_model_name,
    "threshold": best_threshold,
    "roc_auc_test": roc_auc_score(y_test, test_proba),
    "pr_auc_test": average_precision_score(y_test, test_proba),
    "precision_test": precision_score(y_test, test_pred),
    "recall_test": recall_score(y_test, test_pred),
    "f2_test": fbeta_score(y_test, test_pred, beta=2),
    "positive_rate_test": float(y_test.mean()),
    "n_train": int(len(X_train)),
    "n_valid": int(len(X_valid)),
    "n_test": int(len(X_test)),
}

pd.DataFrame([metrics_test])


,model,threshold,roc_auc_test,pr_auc_test,precision_test,recall_test,f2_test,positive_rate_test,n_train,n_valid,n_test
0,RandomForest,0.356208,0.915249,0.886825,0.626866,0.913043,0.836653,0.4,535,115,115


In [12]:

print("Matriz de confusão (teste):")
print(confusion_matrix(y_test, test_pred))
print()
print(classification_report(y_test, test_pred, digits=3))


Matriz de confusão (teste):
[[44 25]
 [ 4 42]]

              precision    recall  f1-score   support

           0      0.917     0.638     0.752        69
           1      0.627     0.913     0.743        46

    accuracy                          0.748       115
   macro avg      0.772     0.775     0.748       115
weighted avg      0.801     0.748     0.749       115



## 11. Importância das variáveis

Para apoiar o storytelling do grupo, calculamos importância por permutação no conjunto de teste.


In [13]:

perm = permutation_importance(
    best_pipe, X_test, y_test,
    scoring="average_precision",
    n_repeats=10,
    random_state=42,
)

importances = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

importances.head(15)


,feature,importance_mean,importance_std
0,Defasagem_2023,0.018590,0.006734
1,INDE_2023,0.015676,0.004692
2,IPS_2023,0.009165,0.003509
3,IPV_2023,0.009054,0.007688
4,Avaliador1_2023,0.007957,0.011633
5,Fase_Ideal_2022,0.005894,0.004567
6,Pedra_2023,0.005268,0.002166
7,Avaliador2_2023,0.005238,0.008357
8,IAN_2023,0.004984,0.007241
9,Instituição_de_ensino_2023,0.003951,0.003393


## 12. Salvando artefatos para o Streamlit

Salvamos:

- o pipeline completo treinado;
- o threshold ótimo;
- metadados úteis para o app.


In [14]:

joblib.dump(best_pipe, "modelo_risco_defasagem_pipeline_com_ipp2022.joblib")

metadata = {
    "model": best_model_name,
    "threshold": best_threshold,
    "target_definition": "target = 1 se Defasagem_2024 < 0",
    "notes": [
        "IPP_2022 reconstruído por imputação KNN supervisionada com base em 2023 e 2024.",
        "Features de 2024 excluídas para evitar leakage.",
        "Escolha de modelo por PR-AUC na validação.",
        "Escolha de threshold por F2 na validação."
    ],
    "metrics_test": metrics_test,
    "feature_columns": list(X.columns),
}

with open("threshold_risco_defasagem_com_ipp2022.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Arquivos salvos com sucesso.")


Arquivos salvos com sucesso.


## 13. Conclusão

A solução de machine learning atendeu ao objetivo do projeto ao prever, com bom desempenho, quais alunos apresentam maior risco de defasagem. O modelo final foi construído com base no histórico dos alunos, avaliado com métricas adequadas e preparado para uso na aplicação do grupo.